<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/15_NL2QRT_with_Semantic_Query_Planning_chinook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Natural Language to Query Runtime (NL2QRT) with Semantic Query Planning

## Overview

This notebook implements a semantic analytics pipeline using:

- SQLite + Chinook
- Knowledge Graph based join planning
- Semantic metrics
- Semantic dimensions
- LLM-based semantic query planning
- Graph-based SQL generation
- Automatic SQL repair
- Time semantics
- Filter semantics


## 1. Install / Import Dependencies

In [ ]:
!pip install openai matplotlib networkx

In [ ]:
import sqlite3
import pandas as pd
import networkx as nx
import json
import matplotlib.pyplot as plt
from openai import OpenAI
from google.colab import userdata

## 2. Download Chinook Database

In [ ]:
!wget https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite

## 3. Connect To Database

In [ ]:
conn = sqlite3.connect(
    "Chinook_Sqlite.sqlite"
)
# SQLite is a file-based database.
# This command allows us to open this file and query it.

## 4. Initialize OpenAI Client

In [ ]:
#define constants
MODEL = "gpt-4.1-mini"
TEMP = 0

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

## 5. Extract Schema

In [ ]:
schema_df = pd.read_sql(
    """
    SELECT
        name,
        sql
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

schema_df


In [ ]:
#Show detailed schema information. Easier to understand !

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
AND name NOT LIKE 'sqlite_%';
""", conn)

all_schema = []

for table in tables["name"]:

    # Column info
    columns = pd.read_sql(f"""
    PRAGMA table_info({table});
    """, conn)

    # Foreign key info
    fks = pd.read_sql(f"""
    PRAGMA foreign_key_list({table});
    """, conn)

    fk_columns = set(fks["from"].tolist()) if not fks.empty else set()

    for _, col in columns.iterrows():

        all_schema.append({
            "table": table,
            "column": col["name"],
            "data_type": col["type"],
            "primary_key": bool(col["pk"]),
            "foreign_key": col["name"] in fk_columns,
            "nullable": not bool(col["notnull"])
        })

schema_details = pd.DataFrame(all_schema)

schema_details

In [ ]:
#View One Table as an easy to understand list
schema_details[
    schema_details["table"] == "MediaType"
]

## 6. Extract Relationships

In [ ]:
relationships = []

for table_name in schema_df["name"]:

    fk_df = pd.read_sql(
        f"PRAGMA foreign_key_list({table_name})",
        conn
    )
    #pd.read_sql(...) is a Pandas function.
    #PRAGMA foreign_key_list(Album) is a built-in SQLite command

    for _, row in fk_df.iterrows():
      # _ indicates row number is ignored . No point getting index as we won't use it

        relationships.append({

            "source_table": table_name,

            "source_column": row["from"],

            "target_table": row["table"],

            "target_column": row["to"]
        })

relationships_df = pd.DataFrame(
    relationships
)

relationships_df

## 7. Build Knowledge Graph

In [ ]:
schema_graph = {}

for _, row in relationships_df.iterrows():

    source = row["source_table"]
    target = row["target_table"]

    if source not in schema_graph:
        schema_graph[source] = []

    schema_graph[source].append(target)

schema_graph

In [ ]:
#Create and Draw the Knowledge Graph

G = nx.DiGraph()

# Add edges
for _, row in relationships_df.iterrows():

    source = row["source_table"]
    target = row["target_table"]

    G.add_edge(source, target)

# Draw
plt.figure(figsize=(10, 8))

pos = nx.spring_layout(G, seed=42)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2000,
    font_size=8,
    arrows=True
)

plt.title("Chinook Schema Knowledge Graph")
plt.show()

## 8. Join Path Finder

**networkx** is a Python library for working with graphs (networks).It understands the database schema as a graph and finds relationship paths between tables.

Example :

```
nx.shortest_path(
    graph,
    source="InvoiceLine",
    target="Genre"
)
```
gives us :

      InvoiceLine -> Track -> Genre


In [ ]:
def find_join_path(
    source_table,
    target_table,
    relationships_df
):

    graph = nx.Graph()

    for _, row in relationships_df.iterrows():

        graph.add_edge(
            row["source_table"],
            row["target_table"]
        )

    try:

        path = nx.shortest_path(
            graph,
            source=source_table,
            target=target_table
        )

        return path

    except:

        return None

## 9. Entity Registry

*    Entity Registry are the tables related to the **business concepts users ask about** .
*    It is not a list of all the tables in Chinook DB.

*    **Example :** If you want to answer questions about artists or albums then you add it here




In [ ]:
entity_registry = {

    "artist": {
        "table": "Artist",
        "display_column": "Name",
        "id_column": "ArtistId",
        "entity_type": "dimension"
    },

    "album": {
        "table": "Album",
        "display_column": "Title",
        "id_column": "AlbumId",
        "entity_type": "dimension"
    },

    "track": {
        "table": "Track",
        "display_column": "Name",
        "id_column": "TrackId",
        "entity_type": "dimension"
    },

    "genre": {
        "table": "Genre",
        "display_column": "Name",
        "id_column": "GenreId",
        "entity_type": "dimension"
    },

    "customer": {
        "table": "Customer",
        "display_column": "LastName",
        "id_column": "CustomerId",
        "entity_type": "dimension"
    },

    "country": {
        "table": "Customer",
        "display_column": "Country",
        "id_column": "CustomerId",
        "entity_type": "dimension"
    },

    "invoice": {
        "table": "Invoice",
        "display_column": "InvoiceId",
        "id_column": "InvoiceId",
        "entity_type": "fact"
    },

    "employee": {
        "table": "Employee",
        "display_column": "LastName",
        "id_column": "EmployeeId",
        "entity_type": "dimension"
    },

    "playlist": {
        "table": "Playlist",
        "display_column": "Name",
        "id_column": "PlaylistId",
        "entity_type": "dimension"
    },

    "mediatype": {
        "table": "MediaType",
        "display_column": "Name",
        "id_column": "MediaTypeId",
        "entity_type": "dimension"
    }

}

## 10. Semantic Metrics

In [ ]:
semantic_metrics = {

    "revenue": {

        "SUM": {

            "expression": (
                "InvoiceLine.UnitPrice * "
                "InvoiceLine.Quantity"
            ),

            "aggregation": "SUM",

            "base_table": "InvoiceLine"
        },

        "AVG": {

            "expression": "Invoice.Total",

            "aggregation": "AVG",

            "base_table": "Invoice"
        }
    },

    "invoice": {

        "COUNT": {
            "expression": "Invoice.InvoiceId",
            "aggregation": "COUNT",
            "base_table": "Invoice"
        },

        "AVG": {
            "expression": "Invoice.Total",
            "aggregation": "AVG",
            "base_table": "Invoice"
        },

        "SUM": {
            "expression": "Invoice.Total",
            "aggregation": "SUM",
            "base_table": "Invoice"
        }
    },

    "track price": {

        "AVG": {

            "expression": "Track.UnitPrice",

            "aggregation": "AVG",

            "base_table": "Track"
        },

        "SUM": {

            "expression": "Track.UnitPrice",

            "aggregation": "SUM",

            "base_table": "Track"
        }
    }
}

## 11. Semantic Time Dimensions

In [ ]:
semantic_dimensions = {

    "month": {

        "expression": (
            "strftime('%Y-%m', "
            "Invoice.InvoiceDate)"
        ),

        "base_table": "Invoice"
    },

    "year": {

        "expression": (
            "strftime('%Y', "
            "Invoice.InvoiceDate)"
        ),

        "base_table": "Invoice"
    }
}

##12. LLM Prompt Builder

Add the entities for which you want to ask questions in the section **AVAILABLE ENTITIES**

In [ ]:
def build_llm_planner_prompt(user_query):

    prompt = f"""
You are a semantic query planner.

Convert the natural language query into
a structured semantic query plan.

Return ONLY valid JSON.

==================================================
AVAILABLE ENTITIES
==================================================

artist
album
track
genre
customer
country
invoice
employee
playlist
mediatype

==================================================
AVAILABLE BUSINESS METRICS
==================================================

revenue
invoice
track price
track

==================================================
AVAILABLE TIME DIMENSIONS
==================================================

month
year

==================================================
AVAILABLE AGGREGATIONS
==================================================

COUNT
COUNT_DISTINCT
SUM
AVG

==================================================
JSON FORMAT
==================================================

{{
    "group_by": "...",
    "metric": "...",
    "aggregation": "...",

    "filters": [
        {{
            "field": "...",
            "value": "..."
        }}
    ],

    "order": "DESC",

    "limit": 10
}}

==================================================
USER QUERY
==================================================

{user_query}
"""

    return prompt


## 13. Semantic Query Planner

In [ ]:
def get_semantic_query_plan(user_query):

    prompt = build_llm_planner_prompt(
        user_query
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[

            {
                "role": "system",

                "content": (
                    "You are a semantic query planner. "
                    "Return ONLY valid JSON."
                )
            },

            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=TEMP
    )

    response_text = (
        response
        .choices[0]
        .message
        .content
    )

    print("RAW LLM RESPONSE:")
    print(response_text)

    cleaned_response = (
        response_text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    query_plan = json.loads(
        cleaned_response
    )

    print("QUERY PLAN AFTER CLEANING:")
    print(query_plan)

    if "filters" not in query_plan:
        query_plan["filters"] = []

    return query_plan

## 14. Query Plan Validation

In [ ]:
def validate_query_plan(query_plan):

    valid_aggregations = [
        "COUNT",
        "COUNT_DISTINCT",
        "SUM",
        "AVG"
    ]

    if (
        query_plan["aggregation"]
        not in valid_aggregations
    ):

        raise ValueError(
            "Invalid aggregation"
        )

    if (
        query_plan["metric"]
        in valid_aggregations
    ):

        print(
            "WARNING: "
            "LLM returned aggregation "
            "as metric"
        )

        query_plan["metric"] = "track"

    return query_plan


##15. Build Join Graph

It supports:

*   **Direct relationships**

    Example:

        InvoiceLine -> Invoice

    It creates a single edge:

        InvoiceLine --- Invoice

*   **Indirect relationships**

    Example:

        InvoiceLine -> Customer

    There is no direct relationship, so the shortest path is:

        InvoiceLine -> Invoice -> Customer

*   **Multiple indirect relationships simultaneously**

    To answer:

        top customers by rock purchases
        
    we need:

        InvoiceLine
        Customer
        Genre

    which expands to:

    ```
                Invoice
                   |
    Customer <- InvoiceLine -> Track -> Genre
    
    ```

A simple linear join path cannot represent this correctly.

Therefore we build a Join Graph (tree-shaped structure) that can represent multiple direct and indirect relationships at the same time.

In [ ]:
def build_join_graph(
    required_tables,
    relationships_df,
     base_table
):

    # required_tables = list(
    #     required_tables
    # )

    # base_table = required_tables[0]
    required_tables = list(required_tables)

    join_edges = []

    added_edges = set() #creates an empty set

    for table in required_tables: #[1:]:

        if table == base_table:
          continue

        path = find_join_path(
            base_table,
            table,
            relationships_df
        )

        print(
            f"PATH: {base_table} "
            f"-> {table}"
        )

        print(path)

        if path is None:
            continue

        for i in range(len(path) - 1):

            left_table = path[i]
            right_table = path[i + 1]

            edge_key = tuple(
                sorted([
                    left_table,
                    right_table
                ])
            )

            if edge_key in added_edges:
                continue

            relation = relationships_df[
                (
                    (
                        relationships_df[
                            "source_table"
                        ] == left_table
                    )
                    &
                    (
                        relationships_df[
                            "target_table"
                        ] == right_table
                    )
                )
                |
                (
                    (
                        relationships_df[
                            "source_table"
                        ] == right_table
                    )
                    &
                    (
                        relationships_df[
                            "target_table"
                        ] == left_table
                    )
                )
            ]

            if relation.empty:
                continue

            relation = relation.iloc[0]

            join_edges.append({

                "left_table": (
                    relation[
                        "source_table"
                    ]
                ),

                "left_column": (
                    relation[
                        "source_column"
                    ]
                ),

                "right_table": (
                    relation[
                        "target_table"
                    ]
                ),

                "right_column": (
                    relation[
                        "target_column"
                    ]
                )
            })

            added_edges.add(
                edge_key
            )

    return {
        "base_table": base_table,
        "edges": join_edges
    }

## 16. Generate JOIN SQL

In [ ]:
def generate_join_sql_from_graph(join_graph):

    join_sql_parts = []

    for edge in join_graph["edges"]:

        left_table = (
            edge["left_table"]
        )

        left_column = (
            edge["left_column"]
        )

        right_table = (
            edge["right_table"]
        )

        right_column = (
            edge["right_column"]
        )

        join_sql_parts.append(
            f"""
JOIN {right_table}
    ON {left_table}.{left_column}
       =
       {right_table}.{right_column}
"""
        )

    join_sql = "\n".join(
        join_sql_parts
    )

    return join_sql

## 17. SQL Generator

In [ ]:
def generate_sql_from_plan(query_plan):

    group_term = query_plan["group_by"]
    metric_term = query_plan["metric"]
    aggregation = query_plan["aggregation"]

    filters = query_plan.get(
        "filters",
        []
    )

    order = query_plan.get(
        "order",
        "DESC"
    )

    limit = query_plan.get(
        "limit",
        10
    )

    if group_term in semantic_dimensions:

        dimension_definition = (
            semantic_dimensions[group_term]
        )

        group_column = (
            dimension_definition[
                "expression"
            ]
        )

        group_table = (
            dimension_definition[
                "base_table"
            ]
        )

    else:

        group_entity = (
            entity_registry[group_term]
        )

        group_table = (
            group_entity["table"]
        )

        group_column = (
            f"{group_table}."
            f"{group_entity['display_column']}"
        )

    metric_entity = entity_registry.get(
        metric_term,
        None
    )

    if metric_term in semantic_metrics:

        metric_definition = (
            semantic_metrics[metric_term]
        )

        metric_config = (
            metric_definition[
                aggregation
            ]
        )

        metric_expression = (
            metric_config["expression"]
        )

        metric_table = (
            metric_config["base_table"]
        )

    else:

        metric_table = (
            metric_entity["table"]
        )

        metric_expression = (
            f"{metric_table}."
            f"{metric_entity['id_column']}"
        )

    required_tables = set()

    required_tables.add(group_table)
    required_tables.add(metric_table)

    for filter_item in filters:

        field = filter_item["field"]

        normalized_field = (
            field
            .split(".")[0]
            .lower()
        )

        if normalized_field in semantic_dimensions:

            dimension_table = (
                semantic_dimensions[
                    normalized_field
                ]["base_table"]
            )

            required_tables.add(
                dimension_table
            )

        elif normalized_field in entity_registry:

            filter_entity = (
                entity_registry[
                    normalized_field
                ]
            )

            required_tables.add(
                filter_entity["table"]
            )

    print("REQUIRED TABLES:")
    print(required_tables)

    # join_graph = build_join_graph(
    #     required_tables,
    #     relationships_df
    # )
    join_graph = build_join_graph(
        required_tables,
        relationships_df,
        metric_table
    )

    print("JOIN GRAPH:")
    print(join_graph)

    join_sql = (
        generate_join_sql_from_graph(
            join_graph
        )
    )

    base_table = (
        join_graph["base_table"]
    )

    aggregation_sql = (
        f"{aggregation}({metric_expression})"
    )

    where_clause = ""

    where_conditions = []

    for filter_item in filters:

        field = filter_item["field"]

        value = str(
            filter_item["value"]
        )

        normalized_field = (
            field
            .split(".")[0]
            .lower()
        )

        if (
            normalized_field
            in semantic_dimensions
        ):

            dimension_definition = (
                semantic_dimensions[
                    normalized_field
                ]
            )

            filter_expression = (
                dimension_definition[
                    "expression"
                ]
            )

            where_conditions.append(
                f"{filter_expression} = '{value}'"
            )

        elif (
            normalized_field
            in entity_registry
        ):

            filter_entity = (
                entity_registry[
                    normalized_field
                ]
            )

            filter_column = (
                f"{filter_entity['table']}."
                f"{filter_entity['display_column']}"
            )

            where_conditions.append(
                f"{filter_column} = "
                f"'{value.title()}'"
            )

    if where_conditions:

        where_clause = (
            "WHERE " +
            " AND ".join(
                where_conditions
            )
        )

    sql = f"""
    SELECT
        {group_column} AS grouping_value,
        {aggregation_sql} AS metric_value

    FROM {base_table}

    {join_sql}

    {where_clause}

    GROUP BY {group_column}

    ORDER BY metric_value {order}

    LIMIT {limit}
    """

    return sql

## 18. SQL Repair Prompt

In [ ]:
def build_sql_repair_prompt(user_query,sql_query,error_message):

    prompt = f"""
You are a SQL repair assistant.

A SQL query failed during execution.

Your task:
- fix the SQL
- preserve the original intent
- return ONLY valid SQL

USER QUERY:
{user_query}

FAILED SQL:
{sql_query}

DATABASE ERROR:
{error_message}
"""

    return prompt

## 19. SQL Repair Function

In [ ]:
def repair_sql_query(user_query,sql_query,error_message):

    repair_prompt = (
        build_sql_repair_prompt(
            user_query,
            sql_query,
            error_message
        )
    )

    response = client.chat.completions.create(

        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a SQL repair "
                    "assistant. Return ONLY SQL."
                )
            },

            {
                "role": "user",
                "content": repair_prompt
            }
        ],
        temperature=TEMP
    )

    repaired_sql = (
        response
        .choices[0]
        .message
        .content
    )

    repaired_sql = (
        repaired_sql
        .replace("```sql", "")
        .replace("```", "")
        .strip()
    )

    return repaired_sql

## 20. Main Orchestrator

In [ ]:
def ask_database(user_query):

    print("=" * 60)
    print("USER QUERY:")
    print(user_query)

    query_plan = get_semantic_query_plan(
        user_query
    )

    query_plan = validate_query_plan(
        query_plan
    )

    print("\n" + "=" * 60)
    print("SEMANTIC QUERY PLAN:")
    print(query_plan)

    sql_query = generate_sql_from_plan(
        query_plan
    )

    print("\n" + "=" * 60)
    print("GENERATED SQL:")
    print(sql_query)

    try:

        result_df = pd.read_sql(
            sql_query,
            conn
        )

        print("\n" + "=" * 60)
        print("SQL EXECUTION SUCCESS")

        return result_df

    except Exception as e:

        print("\n" + "=" * 60)
        print("SQL EXECUTION FAILED")

        error_message = str(e)

        print(error_message)

    repaired_sql = repair_sql_query(

        user_query=user_query,

        sql_query=sql_query,

        error_message=error_message
    )

    print("\n" + "=" * 60)
    print("REPAIRED SQL:")
    print(repaired_sql)

    repaired_result_df = pd.read_sql(
        repaired_sql,
        conn
    )

    print("\n" + "=" * 60)
    print("REPAIRED SQL EXECUTION SUCCESS")

    return repaired_result_df

## 21. Example Queries

In [ ]:
ask_database("top countries by revenue")

In [ ]:
ask_database("highest selling genres")

In [ ]:
ask_database("average invoice by country")

In [ ]:
ask_database("top customers by rock purchases")

In [ ]:
ask_database("monthly revenue trend")

In [ ]:
ask_database("top 5 musicians in 2010")
# all these synomyms would work - artist , singer, band, musician

In [ ]:
ask_database("genres with highest average song price") #synonyms : track , song

In [ ]:
ask_database("artists with the most albums")

In [ ]:
ask_database("highest selling tracks in the Rock genre")

In [ ]:
ask_database("top buyers in Brazil by revenue") #synomyms : customer , buyer, purchaser

In [ ]:
ask_database("top albums")
# ambiguous question ... it returns albums with highest number of tracks

In [ ]:
ask_database("top albums by revenue")

In [ ]:
ask_database("best artist")
# ambiguous question ... it rates artists by revenue

In [ ]:
ask_database("top genres by average invoice") # auto repair of generated SQL

In [ ]:
ask_database("top customers by invoice count")

In [ ]:
ask_database("top employees by customer ")

In [ ]:
ask_database("top 5 playlists by revenue ")

In [ ]:
ask_database("top 5 playlists by track count ")

In [ ]:
ask_database("top media types by track count")

In [ ]:
ask_database("top media types by revenue")

# Further Enhancement Opportunities

1. Multi-metric support :
The plan only allows one metric. Most real analytics questions need two: "top genres by revenue and track count". The JSON plan schema would need a metrics: [] array.

2. Multi-dimensional group-by :
Only one group_by is supported. Questions like "revenue by genre and year" are common and currently impossible.

3. HAVING clause :
No way to filter on aggregated values: "genres with more than 1000 tracks". This is a natural follow-on to the WHERE support you already have.

4. Conversational follow-up :
Right now each call is stateless. Adding conversation memory ("now filter to only 2013", "show the same for albums") would make it feel like a real analytics assistant rather than a one-shot tool.

5. Query explanation to the user :
The system knows exactly how it interpreted the question — which metric, which tables, which joins. Surfacing that ("Interpreted as: SUM(InvoiceLine.UnitPrice × Quantity) grouped by Genre") builds trust and helps users correct misinterpretations without guessing.

6. Auto-visualization :
Results come back as DataFrames. Since the plan already knows group_by type and aggregation, we could automatically choose the right chart (bar for categorical, line for time dimensions) using matplotlib or plotly. This would significantly improve the demo experience.

**Available tools that solve this problem ( most likely better : 😀)**

*    **Snowflake Cortex Analyst** — It uses a YAML semantic model defining metrics, dimensions, and relationships, then generates SQL from it. That's structurally the same pattern as what we built.

*    **Cube.dev** — Defines measures, dimensions, and join paths in a semantic layer, then generates SQL deterministically from it. Almost identical concept to our semantic_metrics + entity_registry + join graph.

*   **dbt Semantic Layer (MetricFlow)** — Defines metrics, entities, and dimensions explicitly; generates SQL from semantic queries. Same separation of concerns we built.